# Questão 2 - Atendimentos Hospitalares

Crie uma solução para um sistema de atendimentos prioritário de um hospital.

### Para o desenvolvimento da solução, considere:

- Identifique a melhor forma de representar os pacientes.
- Identifique a melhor estrutura de dados para representar este problema.
- Pacientes passam inicialmente na triagem e ganham um nível de urgência:
  - baixo
  - médio
  - alto
- Quanto maior o nível de urgência, maior a prioridade do paciente.
- Pacientes idosos (`idade >= 60`) têm prioridade sobre os demais.
- Pacientes são atendidos em ordem de chegada, respeitando suas respectivas prioridades.

### Simulação

Implemente uma simulação do hospital com:

- chegada de 10 pacientes;
- idade e nível de urgência aleatórios;
- apenas um médico em atendimento.

Ao final, exiba a ordem de atendimento de cada paciente de acordo com as regras definidas.


## Resolução

#### 1. Classe Node:
A base da estrutura. Cada No guarda o dado do paciente e um "ponteiro" (proximo) que aponta para o paciente que está atrás dele na fila.

In [3]:
class Nodo:
    def __init__(self, dado):
        self.dado = dado
        self.proximo = None

#### 2. Classe Fila
A classe gerencia a fila guardando referências apenas para o primeiro (quem será atendido) e para o ultimo (onde os novos pacientes entram). Essa implementação garante a complexidade O(1) (instantânea) para adicionar no fim e para remover no começo.

In [ ]:
class Fila:
    def __init__(self):
        self.primeiro = None
        self.ultimo = None
        
    def is_empty(self):
        return self.primeiro is None
        
    def enqueue(self, dado): # Adiciona no fim da fila
        novo_no = Nodo(dado)
        if self.is_empty():
            self.primeiro = novo_no
            self.ultimo = novo_no
        else:
            self.ultimo.proximo = novo_no
            self.ultimo = novo_no
            
    def dequeue(self): # Remove e retorna do início da fila
        if self.is_empty():
            return None
        dado = self.primeiro.dado
        self.primeiro = self.primeiro.proximo
        if self.primeiro is None: # Se a fila ficou vazia
            self.ultimo = None
        return dado


#### 3. Representação do Paciente
Uma classe simples Paciente para guardar as informações. Isso ajuda muito a organizar os dados sem nos perdermos com dicionários ou tuplas soltas.

In [4]:
class Paciente:
    def __init__(self, id_paciente, idade, urgencia):
        self.id = id_paciente
        self.idade = idade
        self.urgencia = urgencia # 'alto', 'medio', 'baixo'
        
    def __repr__(self):
        idoso_str = "Idoso" if self.idade >= 60 else "Não Idoso"
        return f"Paciente {self.id:02d} | Idade: {self.idade:02d} ({idoso_str}) | Urgência: {self.urgencia.upper()}"


#### 4. A Estrutura de Filas Múltiplas
Utilizamos a classe Fila criada antetiormente. A classe Fila é uma implementação de Lista Simplemente Encadeada e funciona perfeitamente como uma fila simples (FIFO), sendo extremamente eficiente (Complexidade de Tempo O(1)) tanto para inserir no final quanto para retirar do começo.

Instanciamos 6 filas diferentes dentro de um dicionário na nossa classe HospitalPrioridade. Cada chave representa um nível de prioridade (de 1 a 6, sendo 1 a mais alta):

In [6]:
class HospitalPrioridade:
    def __init__(self):
        # Dicionário mantendo 6 Filas customizadas, em ordem de prioridade.
        self.filas = {
            1: Fila(), # Alta Urgência + Idoso
            2: Fila(), # Alta Urgência + Não Idoso
            3: Fila(), # Média Urgência + Idoso
            4: Fila(), # Média Urgência + Não Idoso
            5: Fila(), # Baixa Urgência + Idoso
            6: Fila()  # Baixa Urgência + Não Idoso
        }
        
    def _calcular_prioridade(self, paciente):
        if paciente.urgencia == 'alto':
            return 1 if paciente.idade >= 60 else 2
        elif paciente.urgencia == 'medio':
            return 3 if paciente.idade >= 60 else 4
        else: # baixo
            return 5 if paciente.idade >= 60 else 6
            
    def triagem(self, paciente):
        prioridade = self._calcular_prioridade(paciente)
        
        # Enfileira o paciente (preservando ordem de chegada)
        self.filas[prioridade].enqueue(paciente)
        print(f"[TRIAGEM] {paciente} -> Entrou na fila de Prioridade {prioridade}")
        
    def atender_proximo(self):
        for prioridade in range(1, 7):
            if not self.filas[prioridade].is_empty(): # Se a fila tiver alguém
                # Desenfileira (FIFO)
                paciente = self.filas[prioridade].dequeue()
                print(f"[ATENDIMENTO] Médico chamou: {paciente} (Prioridade {prioridade})")
                return paciente
                
        print("[ATENDIMENTO] Não há pacientes na fila no momento.")
        return None


#### 5. Executando a Simulação

Execute a celula abaixo para simular a cehaga de 10 pacientes de forma randomica.

In [8]:
import random

# --- SIMULAÇÃO ---
print("="*65)
print(" INICIANDO TRIAGEM DOS PACIENTES (Ordem de Chegada) ")
print("="*65)

hospital = HospitalPrioridade()
urgencias_possiveis = ['alto', 'medio', 'baixo']

# Chegada de 10 pacientes com idades e urgências aleatórias
for i in range(1, 11):
    idade_aleatoria = random.randint(10, 90)
    urgencia_aleatoria = random.choice(urgencias_possiveis)
    
    novo_paciente = Paciente(id_paciente=i, idade=idade_aleatoria, urgencia=urgencia_aleatoria)
    hospital.triagem(novo_paciente)


print("\n" + "="*65)
print(" INICIANDO ATENDIMENTO MÉDICO (Ordem de Prioridade) ")
print("="*65)

for _ in range(10):
    hospital.atender_proximo()


 INICIANDO TRIAGEM DOS PACIENTES (Ordem de Chegada) 
[TRIAGEM] Paciente 01 | Idade: 38 (Não Idoso) | Urgência: MEDIO -> Entrou na fila de Prioridade 4
[TRIAGEM] Paciente 02 | Idade: 90 (Idoso) | Urgência: ALTO -> Entrou na fila de Prioridade 1
[TRIAGEM] Paciente 03 | Idade: 45 (Não Idoso) | Urgência: MEDIO -> Entrou na fila de Prioridade 4
[TRIAGEM] Paciente 04 | Idade: 67 (Idoso) | Urgência: MEDIO -> Entrou na fila de Prioridade 3
[TRIAGEM] Paciente 05 | Idade: 72 (Idoso) | Urgência: MEDIO -> Entrou na fila de Prioridade 3
[TRIAGEM] Paciente 06 | Idade: 18 (Não Idoso) | Urgência: MEDIO -> Entrou na fila de Prioridade 4
[TRIAGEM] Paciente 07 | Idade: 67 (Idoso) | Urgência: MEDIO -> Entrou na fila de Prioridade 3
[TRIAGEM] Paciente 08 | Idade: 57 (Não Idoso) | Urgência: MEDIO -> Entrou na fila de Prioridade 4
[TRIAGEM] Paciente 09 | Idade: 47 (Não Idoso) | Urgência: BAIXO -> Entrou na fila de Prioridade 6
[TRIAGEM] Paciente 10 | Idade: 45 (Não Idoso) | Urgência: MEDIO -> Entrou na fila 

## Complexidade de algoritmos

Análise assintótica (Notação Big-O) de Tempo para o algoritmo implmentado;

- **Inserção na fila**: Complexidade O(1) constante. Como guardamos uma referência explicita para o `ultimo` nó da fila, não precisamos percorrer a fila inteira para encontrar o final. A inserção consiste apenas em criar um novo nó e redirecionar dois ponteiros de memória. O tempo de execuação é constante quer a fila tenha 10 ou 1000 pacientes.

- **Remoção na fila**: Complexidade O(1) constante. A remoção sempre ocorre no início da fila. O algoritmo acessa o `primeiro` nó, salva seu dados e atualiza o ponteiro `primeiro` para apontar para o próximo nó da lista. Não há laços de repetição, resultando em um tempo constante.

- **Triagem na classe HospitalPrioridade**: Complexidade O(1) constante. O cáculo da prioridade possui um número fixo e pequeno de condicionais `if/else`. Após determinar em qual fila o paciente deve entrar, o algoritmo invoca a operação enqueue (enfileirar) que é O(1), ou seja, dar entrada de um paciente no hospital tem operação instantânea.

- **Atender o próximo na classe HospitalPrioridade**: Complexidade O(1) constante. Ao chamar o próximo paciente, o algoritmo faz um looop para verificar as filas, que neste caso roda no **máximo 6 vezes**, independente do número de pacientes no hospital. Sendo 6 uma constante, o tempo de busca na fila não vazia e a remoção rodam em O (1).

Desta forma, concluímos que o algoritmo de **múltiplas filas encadeadas** oferece eficiência computacional para o problema proposto, rodando as operações fundamentais em **tempo constante O(1)**.
